# HearAssist — Transcription de la darija avec Whisper

Ce notebook suit les étapes du projet :

1. **Préparation** : GPU, installation, code et token Hugging Face
2. **Exploration** du dataset `adiren7/darija_speech_to_text`
3. **Référence sans entraînement** : Whisper de base et modèles darija publics
4. **Test rapide** du fine-tuning (quelques minutes)
5. **Fine-tuning réel** de `whisper-small` (à lancer avec *Save Version → Save & Run All*)
6. **Conversion** pour faster-whisper et **évaluation finale**

**Réglages Kaggle (panneau de droite) :** Accelerator = **GPU T4 x2** · Internet = **On** ·
Input = le dataset Kaggle contenant le dossier `darija_asr`.

## 1. Préparation

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # un seul GPU : plus simple et plus stable
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q "transformers>=5.13" "datasets>=3.0" "peft>=0.13" accelerate jiwer soundfile ctranslate2 faster-whisper tiktoken sentencepiece
# torchao préinstallé sur Kaggle (trop ancien) fait planter LoRA (peft) ; on n'en a pas besoin
!pip uninstall -y -q torchao

In [ ]:
# Copie du code (dataset Kaggle contenant darija_asr/) vers le dossier de travail
import glob, shutil
src = glob.glob("/kaggle/input/**/finetune_whisper.py", recursive=True)
assert src, "Ajoute le dataset Kaggle contenant le dossier darija_asr (Add Input)"
CODE = "/kaggle/working/darija_asr"
shutil.copytree(os.path.dirname(src[0]), CODE, dirs_exist_ok=True)
os.chdir(CODE)
print("Code prêt dans", CODE, ":", sorted(os.listdir(CODE)))

In [ ]:
# Token Hugging Face (facultatif pour adiren7, nécessaire pour pousser un modèle sur le Hub)
# Ajoute-le dans Add-ons -> Secrets sous le nom HF_TOKEN
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    print("Connecté à Hugging Face")
except Exception as e:
    print("Pas de token HF (ce n'est pas bloquant pour commencer) :", type(e).__name__)

## 2. Exploration du dataset
Les données sont téléchargées directement depuis Hugging Face (~1,6 Go, 1 à 2 minutes).

In [ ]:
import sys
sys.path.insert(0, CODE)
from data import load_darija, decode_audio, clip_duration
from text_norm import normalize

train = load_darija("train")
val = load_darija("validation")
print(train)
print(val)

In [ ]:
import numpy as np
for name, ds in [("train", train), ("validation", val)]:
    dur = np.array(list(ds["end_time"])) - np.array(list(ds["start_time"]))
    chars = np.array([len(t) for t in ds["text"]])
    print(f"{name:10s} | {len(ds):5d} clips | {dur.sum()/3600:5.1f} h | durée médiane {np.median(dur):.1f} s "
          f"| clips < 5 s : {(dur < 5).sum()} | caractères/clip : {chars.mean():.0f}")

In [ ]:
# Écouter quelques exemples et lire leur transcription
from IPython.display import Audio, display
for i in [0, 100, 2000]:
    ex = train[i]
    print(f"--- exemple {i} ({clip_duration(ex):.1f} s)")
    print(ex["text"])
    display(Audio(decode_audio(ex["audio"]), rate=16000))

## 3. Référence sans entraînement
On mesure le WER et le CER **avant** tout fine-tuning, sur 200 clips de validation.
C'est le point de comparaison du rapport.

Les modèles publics servent de comparaison avec l'existant. Si l'un d'eux ne se charge pas, retire-le de la liste.

In [ ]:
N = 200
BASELINES = [
    ("faster-whisper", "small"),
    ("faster-whisper", "medium"),
    ("faster-whisper", "large-v3-turbo"),
    ("transformers", "ychafiqui/whisper-small-darija"),
    ("transformers", "anaszil/whisper-large-v3-turbo-darija"),
]
for backend, model in BASELINES:
    print(f"\n===== {backend} : {model}")
    !python eval_asr.py --backend {backend} --model {model} --max-samples {N} --out-dir /kaggle/working/results

In [ ]:
import glob, json
import pandas as pd
res = [json.load(open(f)) for f in glob.glob("/kaggle/working/results/*_metrics.json")]
table = pd.DataFrame(res)[["model", "backend", "n_samples", "wer", "cer", "rtf", "seconds_per_clip"]].sort_values("wer")
table

In [ ]:
# Analyse d'erreurs : comparer quelques références et prédictions
preds = pd.read_csv(sorted(glob.glob("/kaggle/working/results/small_*_predictions.csv"))[0])
for _, r in preds.head(5).iterrows():
    print("REF :", r["ref_norm"][:200]); print("HYP :", r["hyp_norm"][:200]); print()

## 4. Test rapide du fine-tuning
Vérifie en quelques minutes que toute la chaîne fonctionne sur le GPU :
fine-tuning complet, LoRA, conversion CTranslate2, puis évaluation du modèle converti.

In [ ]:
!python finetune_whisper.py --base-model openai/whisper-tiny --max-train-samples 64 --eval-samples 16 \
    --max-steps 10 --eval-steps 10 --warmup-steps 0 --output-dir /kaggle/working/smoke
!python finetune_whisper.py --base-model openai/whisper-tiny --lora --max-train-samples 64 --eval-samples 16 \
    --max-steps 10 --eval-steps 10 --warmup-steps 0 --output-dir /kaggle/working/smoke_lora
!python convert_ct2.py --model /kaggle/working/smoke/final --out /kaggle/working/smoke/ct2
!python eval_asr.py --backend faster-whisper --model /kaggle/working/smoke/ct2 --max-samples 16 --out-dir /kaggle/working/smoke_results

## 5. Fine-tuning réel de whisper-small
⚠️ Plusieurs heures. Lance-le avec **Save Version → Save & Run All** pour qu'il continue navigateur fermé,
ou passe `RUN_TRAINING = True` et exécute la cellule en session interactive.

- Les checkpoints sont écrits dans `/kaggle/working` (conservé à la fin de l'exécution).
- Avec `HUB_REPO` renseigné (token HF requis), chaque checkpoint est aussi poussé sur le Hub.
  Après une coupure, ajoute `--resume` pour reprendre.

In [ ]:
RUN_TRAINING = False
HUB_REPO = ""          # ex. "ton-pseudo/whisper-small-darija" (laisser vide pour ne rien pousser)
OUT = "/kaggle/working/whisper-small-darija"

if RUN_TRAINING:
    hub = f"--push-to-hub {HUB_REPO}" if HUB_REPO else ""
    !python finetune_whisper.py --base-model openai/whisper-small --output-dir {OUT} \
        --batch-size 16 --grad-accum 1 --max-steps 3000 --eval-steps 250 --eval-samples 200 {hub}

## 6. Conversion et évaluation finale

In [ ]:
if os.path.exists(f"{OUT}/final"):
    !python convert_ct2.py --model {OUT}/final --out {OUT}/ct2 --quantization float16
    !python eval_asr.py --backend faster-whisper --model {OUT}/ct2 --max-samples 0 --out-dir /kaggle/working/results
else:
    print("Pas encore de modèle entraîné : lance d'abord l'étape 5.")